# الفصل 36: مُشفِّرات الرؤية: ViT وCLIP

> قبل أن يستطيع أي LLM "رؤية" صورة، يجب تحويلها إلى تسلسل من المتجهات، تمامًا كما يُحوَّل النص إلى تسلسل من توكنز التضمين.

- **الجزء:** 8 — النماذج متعددة الوسائط
- **الفصل:** 36
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 35: أساسيات النماذج متعددة الوسائط
- **الوسوم:** ViT، CLIP، SigLIP، Contrastive Learning

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch36-vision-encoders.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-08-multimodal%2Fch36-vision-encoders.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-08-multimodal/ch36-vision-encoders.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch36-vision-encoders.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch36-vision-encoders.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch36-vision-encoders.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch36-vision-encoders.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-08-multimodal%2Fch36-vision-encoders.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-08-multimodal/ch36-vision-encoders.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch36-vision-encoders.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch36-vision-encoders.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch36-vision-encoders.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> قبل أن يستطيع أي LLM "رؤية" صورة، يجب تحويلها إلى تسلسل من المتجهات، تمامًا كما يُحوَّل النص إلى تسلسل من توكنز التضمين.

> **بيانات الفصل**
>
> - **الجزء:** 8 — النماذج متعددة الوسائط
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 35: أساسيات النماذج متعددة الوسائط


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم CLIP: محاذاة الرؤية واللغة عبر التعلّم التبايني
- فهم ViT
- فهم CLIP


## 1. لماذا يهمّك هذا الموضوع؟

🟢 قبل أن يستطيع أي LLM "رؤية" صورة، يجب تحويلها إلى تسلسل من المتجهات، تمامًا كما يُحوَّل النص إلى تسلسل من توكنز التضمين. Vision Transformer (ViT) هو الحل الذي طبّق فكرة Transformer نفسها (المُصمَّمة أصلًا للنص) على الصور بنجاح باهر.


## 2. الخريطة الذهنية


```text
مُشفِّر الرؤية
│
├── ViT (تصنيف/تمثيل)
│   صورة → رقع 16×16 → إسقاط خطي → [CLS] + PosEmb
│        → Transformer Encoder → [CLS] للتصنيف
│                              → كل الرقع لـ VLM
│
└── CLIP (محاذاة نص-صورة)
    صورة → ViT → Embedding صورة ─┐
                                 ├→ تشابه جيب التمام → خسارة تباينية
    نص   → Text Encoder → Embedding نص ─┘
```


ViT يجيب على سؤال «كيف نمثّل صورة؟»، وCLIP يجيب على سؤال «كيف نجعل تمثيل
الصورة قابلًا للمقارنة مع النص؟». الأول مُشفِّر أحادي الوسيط، والثاني مُشفِّر
مزدوج يُدرَّب على أزواج (صورة، نص).


## 3. الشرح المتدرج

الفكرة الجوهرية: قسِّم الصورة إلى "رقع" (Patches) ثابتة الحجم (مثلًا 16×16 بكسل)، عامل كل رقعة كأنها "توكن"، أضف Positional Embedding، ثم مرِّرها عبر Transformer Encoder قياسي تمامًا (نفس معمارية BERT!).


In [ ]:
"""
Vision Transformer (ViT) من الصفر — التطبيق الكامل
"""

import torch
import torch.nn as nn

class PatchEmbedding(nn.Module):
    """تحويل الصورة إلى تسلسل من تضمينات الرقع (Patches)"""

    def __init__(self, image_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.num_patches = (image_size // patch_size) ** 2

        # طبقة Conv2D بخطوة (stride) مساوية لحجم الرقعة = تقسيم + إسقاط خطي في خطوة واحدة
        self.projection = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size, stride=patch_size
        )

    def forward(self, x):
        # x: [batch, channels, height, width]
        x = self.projection(x)          # [batch, embed_dim, H/patch, W/patch]
        x = x.flatten(2)                # [batch, embed_dim, num_patches]
        x = x.transpose(1, 2)           # [batch, num_patches, embed_dim]
        return x

class VisionTransformer(nn.Module):
    """ViT كامل: Patch Embedding + [CLS] + Positional Embedding + Transformer Encoder"""

    def __init__(self, image_size=224, patch_size=16, in_channels=3,
                 embed_dim=768, num_layers=12, num_heads=12, mlp_ratio=4.0,
                 num_classes=1000, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(image_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches

        # توكن [CLS] قابل للتعلّم، يُمثّل تلخيصًا لكامل الصورة (كما في BERT)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # تضمين موضعي قابل للتعلّم لكل رقعة + توكن [CLS]
        self.pos_embedding = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            dropout=dropout, activation="gelu", batch_first=True,
            norm_first=True,  # Pre-LN كما في المعماريات الحديثة
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

        nn.init.trunc_normal_(self.pos_embedding, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x, return_patch_features=False):
        batch_size = x.shape[0]
        patches = self.patch_embed(x)                        # [batch, num_patches, embed_dim]

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        tokens = torch.cat([cls_tokens, patches], dim=1)      # [batch, num_patches+1, embed_dim]
        tokens = tokens + self.pos_embedding
        tokens = self.dropout(tokens)

        encoded = self.encoder(tokens)
        encoded = self.norm(encoded)

        if return_patch_features:
            # مهم جدًا لاحقًا عند ربط ViT بـ LLM: نحتاج تمثيلات كل الرقع وليس فقط [CLS]
            return encoded[:, 1:, :]   # [batch, num_patches, embed_dim]

        cls_output = encoded[:, 0]     # للتصنيف: نستخدم [CLS] فقط
        logits = self.head(cls_output)
        return logits


> 💡 نقطة حاسمة لفهم VLMs لاحقًا: عند استخدام ViT كمُشفِّر رؤية ضمن نموذج Vision-Language، لا نستخدم فقط تمثيل [CLS] — بل نستخدم كامل تسلسل تمثيلات الرقع (مثلًا 196 رقعة لصورة 224×224 برقع 16×16)، لأن LLM يحتاج معلومات مكانية تفصيلية للإجابة على أسئلة دقيقة حول الصورة.

### 3.1 CLIP: محاذاة الرؤية واللغة عبر التعلّم التبايني

🟡 الفكرة الثورية لـ CLIP (Contrastive Language-Image Pretraining): بدل تدريب مُصنِّف صور تقليدي (يحتاج تصنيفات مُحدَّدة مسبقًا)، دُرِّب CLIP على 400 مليون زوج (صورة، وصف نصي) مُجمَّعة من الإنترنت، بهدف تبايني (Contrastive): اجعل تمثيل الصورة قريبًا من تمثيل وصفها الصحيح، وبعيدًا عن أوصاف الصور الأخرى في نفس الدفعة.

معادلة الخسارة التبايتية المتناظرة (Symmetric Contrastive Loss):

$$
\mathcal{L} = \frac{1}{2}\left[
\frac{1}{N}\sum_{i=1}^{N}
\log \frac{\exp(\mathrm{sim}(I_i,T_i)/\tau)}
{\sum_{j=1}^{N}\exp(\mathrm{sim}(I_i,T_j)/\tau)}
+
\frac{1}{N}\sum_{i=1}^{N}
\log \frac{\exp(\mathrm{sim}(T_i,I_i)/\tau)}
{\sum_{j=1}^{N}\exp(\mathrm{sim}(T_i,I_j)/\tau)}
\right]
$$


In [ ]:
"""
CLIP: التعلّم التبايني بين الصور والنصوص — تطبيق كامل
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

class CLIPModel(nn.Module):
    def __init__(self, vision_encoder, text_encoder, embed_dim=512,
                 vision_feature_dim=768, text_feature_dim=768):
        super().__init__()
        self.vision_encoder = vision_encoder   # مثلًا ViT بدون رأس تصنيف
        self.text_encoder = text_encoder       # مثلًا Transformer Encoder نصي

        # إسقاط خطي لتوحيد أبعاد التمثيلين في فضاء مشترك واحد
        self.vision_projection = nn.Linear(vision_feature_dim, embed_dim, bias=False)
        self.text_projection = nn.Linear(text_feature_dim, embed_dim, bias=False)

        # درجة حرارة قابلة للتعلّم (تُهيَّأ لوغاريتميًا لضمان الاستقرار)
        self.logit_scale = nn.Parameter(torch.ones([]) * torch.log(torch.tensor(1 / 0.07)))

    def encode_image(self, images):
        features = self.vision_encoder(images, return_patch_features=False)
        projected = self.vision_projection(features)
        return F.normalize(projected, dim=-1)   # تطبيع على الكرة الواحدية

    def encode_text(self, input_ids, attention_mask):
        features = self.text_encoder(input_ids, attention_mask)  # تمثيل جملة مُجمَّع
        projected = self.text_projection(features)
        return F.normalize(projected, dim=-1)

    def forward(self, images, input_ids, attention_mask):
        image_embeds = self.encode_image(images)       # [batch, embed_dim]
        text_embeds = self.encode_text(input_ids, attention_mask)

        logit_scale = self.logit_scale.exp()
        # مصفوفة تشابه بين كل صورة وكل نص في الدفعة [batch, batch]
        logits_per_image = logit_scale * image_embeds @ text_embeds.T
        logits_per_text = logits_per_image.T

        return logits_per_image, logits_per_text

def clip_contrastive_loss(logits_per_image, logits_per_text):
    batch_size = logits_per_image.shape[0]
    labels = torch.arange(batch_size, device=logits_per_image.device)
    # القطر الرئيسي هو الأزواج الصحيحة (صورة i مع نصها الصحيح i)

    loss_image = F.cross_entropy(logits_per_image, labels)
    loss_text = F.cross_entropy(logits_per_text, labels)
    return (loss_image + loss_text) / 2

def clip_zero_shot_classify(clip_model, image, class_names: list, tokenizer, device="cuda"):
    """تصنيف صورة بدون أي تدريب إضافي (Zero-shot) باستخدام CLIP"""
    prompts = [f"صورة لـ {name}" for name in class_names]
    text_inputs = tokenizer(prompts, padding=True, return_tensors="pt").to(device)

    with torch.no_grad():
        image_embed = clip_model.encode_image(image.unsqueeze(0).to(device))
        text_embeds = clip_model.encode_text(
            text_inputs["input_ids"], text_inputs["attention_mask"]
        )
        similarities = (image_embed @ text_embeds.T).squeeze(0)
        probs = F.softmax(similarities * clip_model.logit_scale.exp(), dim=0)

    best_idx = probs.argmax().item()
    return class_names[best_idx], probs[best_idx].item()


> 💡 SigLIP كتطوير حديث: يستبدل SigLIP (من Google) دالة Softmax التبايتية (التي تحتاج حساب تطبيع عبر كامل الدفعة، مكلف للدفعات الكبيرة) بدالة Sigmoid Loss أبسط تُعامل كل زوج (صورة، نص) كمسألة تصنيف ثنائي مستقلة ("هل هذا الزوج متطابق؟ نعم/لا")، مما يُمكّن استخدام دفعات تدريب أكبر بكثير بكفاءة حسابية أعلى.


## 4. مثال محلول خطوة بخطوة

**الهدف:** حساب عدد الرقع وحجم التسلسل، ثم تطبيق خسارة CLIP التباينية على
دفعة صغيرة.


In [ ]:
import torch
import torch.nn.functional as F

def patch_count(image_size: int, patch_size: int) -> int:
    assert image_size % patch_size == 0, "يجب أن يقبل الحجم القسمة على حجم الرقعة"
    return (image_size // patch_size) ** 2

for size in (224, 336, 448):
    n = patch_count(size, 16)
    print(f"{size}×{size} / 16 = {n} رقعة (+1 لـ[CLS] = {n + 1})")

# خسارة CLIP التباينية على دفعة من 2
def clip_loss(image_emb, text_emb, temperature=0.07):
    image_emb = F.normalize(image_emb, dim=-1)
    text_emb = F.normalize(text_emb, dim=-1)
    logits = image_emb @ text_emb.T / temperature
    labels = torch.arange(len(image_emb))
    loss_i = F.cross_entropy(logits, labels)      # صورة → نص
    loss_t = F.cross_entropy(logits.T, labels)    # نص → صورة
    return (loss_i + loss_t) / 2

image_emb = torch.tensor([[1.0, 0.2, 0.0], [0.1, 1.0, 0.1]])
text_emb = torch.tensor([[0.9, 0.3, 0.1], [0.2, 0.8, 0.2]])
print("CLIP loss:", clip_loss(image_emb, text_emb).item())


**القراءة:** عند درجة حرارة صغيرة (0.07) تصبح الخسارة شديدة الحساسية لأي
تشابه غير قطري. هذا يفسّر لماذا تحتاج CLIP إلى دفعات كبيرة (batch size كبير):
كل دفعة تُوفّر «أمثلة سلبية» أكثر، فتتحسن المحاذاة.


## 5. نصائح وحيل

- **استخدم ViT مُدرَّبًا مسبقًا** (CLIP/DINO/ImageNet)؛ التدريب من الصفر يحتاج بيانات ضخمة.
- **اختر حجم رقعة مناسبًا:** 16×16 افتراضي، 14×14 شائع في CLIP، ورقع أصغر للتفاصيل الدقيقة.
- **طبّق نفس preprocessing** الذي استُخدم في تدريب المُشفِّر (mean/std، resize، crop).
- **استخدم كل الرقع في VLM**، وليس [CLS] فقط؛ الـLLM يحتاج معلومات مكانية.
- **جمّد المُشفِّر أولًا**، ثم افتحه جزئيًا فقط عند الحاجة.
- **راقب طول التسلسل:** 224/16 = 196 رمزًا، 448/16 = 784 رمزًا؛ كل زيادة تضاعف كلفة الانتباه.
- **استخدم mixed precision** (bf16/fp16) لتسريع التدريب والاستدلال.
- **قيّم جودة التمثيل** على مهام نقل (linear probe) قبل بناء VLM كامل.


## 6. أخطاء شائعة ومصائد

- **تدريب ViT من الصفر على بيانات صغيرة:** يتفوق CNN عليه بسبب التحيّز الاستقرائي.
- **نسيان التطبيع:** إدخال صور بقيم 0–255 بدل 0–1 أو mean/std الصحيح يدمّر الأداء.
- **استخدام [CLS] فقط في VLM:** يفقد النموذج المعلومات المكانية الدقيقة.
- **تغيير حجم الرقعة بعد التدريب:** يغيّر عدد الرقع ومعنى التضمينات الموضعية.
- **تجاهل نسبة الأبعاد:** قصّ (crop) عدواني قد يحذف أجزاء مهمة من الصورة.
- **الخلط بين ViT التصنيفي وCLIP:** CLIP مُدرَّب على أزواج نص-صورة، وViT التصنيفي على فئات ثابتة.
- **إهمال تكلفة الذاكرة:** خريطة الانتباه لـ784 رمزًا أكبر بأربع مرات من 196.
- **تقييم CLIP على مهمة توليد:** CLIP لا يولّد نصًا؛ استخدمه للاسترجاع/التصنيف.


## 7. جدول مقارنة

| المعيار | ViT (تصنيف قياسي) | CLIP (تبايني) |
|---|---|---|
| بيانات التدريب | صور مُصنَّفة يدويًا (ImageNet) | أزواج (صورة، نص) من الإنترنت |
| الهدف | تصنيف إلى فئات محدودة مسبقًا | محاذاة تمثيل صوري-نصي عام |
| قابلية التعميم لفئات جديدة | ❌ محدودة (يحتاج إعادة تدريب الرأس) | ✅ ممتازة (Zero-shot) |
| الاستخدام في VLMs الحديثة | أساس معماري (Backbone) | أساس معماري + تهيئة محاذاة قوية |


## 8. ملخص أفضل الممارسات

- استخدم مُشفِّر رؤية مُدرَّب مسبقًا (CLIP/SigLIP) دائمًا بدل تدريب ViT من الصفر لمشاريع VLM استخدم تمثيلات كل الرقع (patch-level) لا [CLS] فقط عند الربط بـ LLM جمِّد مُشفِّر الرؤية (على الأقل في المرحلة الأولى من تدريب VLM) لتوفير الموارد والحفاظ على الاستقرار


## 9. أسئلة مقابلات

س: لماذا يُعتبر CLIP نقطة تحوّل مهمة في عالم Multimodal؟

ج: لأنه أثبت أن التعلّم من بيانات ضخمة وغير مُنظَّمة (أزواج صورة-نص من الإنترنت) عبر هدف تبايني بسيط يُنتج تمثيلات بصرية عامة الغرض وقوية جدًا، قابلة للاستخدام Zero-shot في مهام لم يُدرَّب عليها صراحة أبدًا — وهو حجر الأساس الذي بُنيت عليه تقريبًا كل النماذج متعددة الوسائط الحديثة.

س: ما الفائدة من استخدام الرقع (Patches) بدل البكسلات الفردية في ViT؟

ج: معالجة كل بكسل كتوكن منفرد ستُنتج تسلسلًا هائل الطول (224×224=50176 توكن لصورة واحدة فقط)، مما يجعل التعقيد التربيعي لـ Self-Attention غير قابل للتطبيق عمليًا؛ تقسيم الصورة لرقع (مثلًا 16×16) يُقلّل طول التسلسل إلى 196 توكن فقط، وهو رقم عملي تمامًا.


## 10. تمرين عملي

نفّذ PatchEmbedding وVisionTransformer أعلاه، ومرِّر صورة عشوائية (tensor بشكل [1, 3, 224, 224]) تحقق من أن الناتج بالشكل الصحيح [1, 196, 768] عند return_patch_features=True.


## 11. مراجع للتعمّق

- "An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale" (ViT، Dosovitskiy et al., 2021)
- "Learning Transferable Visual Models From Natural Language Supervision" (CLIP، Radford et al., 2021)
- "Sigmoid Loss for Language Image Pre-Training" (SigLIP، Zhai et al., 2023)


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — النماذج متعددة الوسائط / الفصل 36: مُشفِّرات الرؤية: ViT وCLIP.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
